# Scenario Playground Step 8: Point-in-Time Matrix

Objective: combine the completed event-aware shock, state, and 21-session target tables into one raw model-value table while keeping timing provenance and audit-only fields separate.

Success means one deterministic row per usable sample, no future-filled values, no missing model values, exact column provenance, frozen chronological periods, and no fitted preprocessing or model.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.model_matrix import (
    FINAL_FEATURE_NAMES,
    build_model_matrix_artifacts,
    persist_model_matrix_artifacts,
)

artifacts = build_model_matrix_artifacts()
output_paths = persist_model_matrix_artifacts(artifacts)
matrix = artifacts["matrix"]
provenance = artifacts["provenance"]
source_row_audit = artifacts["source_row_audit"]
split_manifest = artifacts["split_manifest"]
summary = artifacts["summary"]
print({"rows": len(matrix), "features": len(FINAL_FEATURE_NAMES), "targets": summary["target_count"], "matrix_columns": len(matrix.columns)})
output_paths

## Exact column-role separation

The model-value CSV contains only `sample_date`, raw features, and targets. Dates, event identities, source-timing decisions, eligibility checks, and split controls live in separate artifacts and cannot be selected accidentally as model features.

In [ ]:
role_counts = {
    "row_key": 1,
    "features": summary["final_feature_count"],
    "targets": summary["target_count"],
    "provenance_columns_separate": len(provenance.columns),
    "source_audit_columns_separate": len(source_row_audit.columns),
}
print(role_counts)
print(pd.Series(summary["feature_family_counts"], name="feature_count"))
artifacts["column_catalog"].head(12)

## Missingness and audit-driven exclusions

The July 2013 CPI event has no compatible expectation strictly before release. Its row is removed rather than assigning a false zero or using future information. The final matrix also omits the oil percentage-volatility feature rejected by Step 9; no valid pandemic row is deleted.

In [ ]:
display(artifacts["row_exclusions"])
display(artifacts["feature_exclusions"])
print("Total final missing values:", int(matrix.isna().sum().sum()))
assert len(matrix) == 584
assert matrix.isna().sum().sum() == 0

## Chronological periods and target-overlap purge

Period boundaries are the frozen Step 4 canonical-calendar 60/20/20 dates. Purging compares each row's actual `target_end_date` with the next period's first target baseline. It does not remove 21 irregular event-aware rows.

In [ ]:
print(split_manifest["modeling_role"].value_counts().sort_index())
purged = split_manifest[split_manifest["purged_for_target_overlap"]]
display(purged)
pd.DataFrame(summary["split"]["frozen_periods"]).T

## Step 8 acceptance

These checks confirm one-to-one assembly, exact 21-session labels, point-in-time boundaries, raw values, complete provenance, and split safety. They do not fit preprocessing or train a model.

In [ ]:
validation = pd.Series(summary["validations"], name="passed")
print(validation.to_string())
assert validation.all()
print("STEP_8_COMPLETE_PENDING_STEP_9_REAUDIT")